# USL Championship — Game Data

Match results, xG, xPoints, analysis using the ASA public API.

In [9]:
from datetime import date
from typing import Final

import numpy as np
import pandas as pd
import pgeocode
import requests
from geopy.distance import geodesic
from itscalledsoccer.client import AmericanSoccerAnalysis

pd.options.display.max_columns = 999
pd.set_option("expand_frame_repr", False)

asa_client = AmericanSoccerAnalysis()

LEAGUE: Final[str] = "uslc"

## Data Fetch

In [10]:
games = asa_client.get_games(leagues=[LEAGUE])
games["date_time_est"] = (
    pd.to_datetime(games["date_time_utc"])
    .dt.tz_convert("US/Eastern")
    .dt.tz_localize(None)
)
games = games.drop(columns=["date_time_utc"])

game_xg = asa_client.get_game_xgoals(leagues=[LEAGUE])
game_xg["date_time_est"] = (
    pd.to_datetime(game_xg["date_time_utc"])
    .dt.tz_convert("US/Eastern")
    .dt.tz_localize(None)
)
game_xg = game_xg.drop(columns=["date_time_utc"])

## Data Cleaning

Resolve IDs to names (teams, managers, referees, stadiums), derive match attributes,
filter to regular season, merge xG, and compute away travel distance.

In [11]:
teams = asa_client.get_teams(leagues=[LEAGUE])
team_map: dict[str, str] = dict(zip(teams["team_id"], teams["team_abbreviation"]))

for _col, _new in [("home_team_id", "home_team"), ("away_team_id", "away_team")]:
    games[_new] = games[_col].map(team_map)
    game_xg[_new] = game_xg[_col].map(team_map)
games = games.drop(columns=["home_team_id", "away_team_id"])
game_xg = game_xg.drop(columns=["home_team_id", "away_team_id"])

managers = asa_client.get_managers(leagues=[LEAGUE])
manager_map: dict[str, str] = dict(
    zip(managers["manager_id"], managers["manager_name"])
)
games["home_manager"] = games["home_manager_id"].map(manager_map)
games["away_manager"] = games["away_manager_id"].map(manager_map)
games = games.drop(columns=["home_manager_id", "away_manager_id"])

referees = asa_client.get_referees(leagues=[LEAGUE])
referee_map: dict[str, str] = dict(
    zip(referees["referee_id"], referees["referee_name"])
)
games["referee"] = games["referee_id"].map(referee_map)
games = games.drop(columns=["referee_id"])

stadiums = asa_client.get_stadia(leagues=[LEAGUE])
stadium_key = stadiums[
    ["stadium_id", "stadium_name", "city", "province", "postal_code"]
].drop_duplicates()
games = (
    games.merge(stadium_key, on="stadium_id", how="left")
    .drop(columns=["stadium_id"])
    .rename(columns={
        "stadium_name": "stadium",
        "province": "state",
        "postal_code": "zip",
    })
    .reset_index(drop=True)
)

In [12]:
def assign_result(
    df: pd.DataFrame, home_col: str, away_col: str, new_col: str = "result"
) -> pd.DataFrame:
    """Return df with a new column: home team name, away team name, or 'DRAW'."""
    result = np.select(
        [df[home_col] > df[away_col], df[home_col] < df[away_col]],
        [df["home_team"], df["away_team"]],
        default="DRAW",
    )
    return df.assign(**{new_col: result})


games = assign_result(games, "home_score", "away_score")
games["date"] = games["date_time_est"].dt.date
games["time_est"] = games["date_time_est"].dt.time
games["day_of_week"] = games["date_time_est"].dt.day_name()
games["matchup"] = games["home_team"] + "v" + games["away_team"]

games = games[~games["knockout_game"].astype(bool)]
games = games[games["status"] == "FullTime"]

game_xg_temp = game_xg[
    [
        "game_id",
        "home_team_xgoals",
        "away_team_xgoals",
        "home_xpoints",
        "away_xpoints",
    ]
].round(2)
games = games.merge(game_xg_temp, on="game_id", how="left")

for col in ["home_score", "away_score", "expanded_minutes", "attendance"]:
    games[col] = pd.to_numeric(games[col], errors="coerce").fillna(0).astype(int)

games["goal_diff"] = games["home_score"] - games["away_score"]
games["xg_diff"] = (games["home_team_xgoals"] - games["away_team_xgoals"]).round(2)
games = assign_result(games, "home_team_xgoals", "away_team_xgoals", "xresult")

# Points are only meaningful for regular-season games. Guard with _is_regular
# so knockout games (if included later) receive NaN instead of misleading values.
_is_regular = ~games["knockout_game"].astype(bool)
games["home_pts"] = np.where(
    _is_regular,
    np.select(
        [games["home_score"] > games["away_score"], games["home_score"] == games["away_score"]],
        [3, 1],
        default=0,
    ),
    np.nan,
)
games["away_pts"] = np.where(
    _is_regular,
    np.select(
        [games["away_score"] > games["home_score"], games["away_score"] == games["home_score"]],
        [3, 1],
        default=0,
    ),
    np.nan,
)
games[["home_pts", "away_pts"]] = games[["home_pts", "away_pts"]].astype("Int64")

# Per-team, per-season metrics: season_pts, game_number, days_rest, ppg.
# Build long-format appearances, sort chronologically within (team, season),
# then compute each metric via groupby before merging back.
_home_apps = games[["game_id", "season_name", "date", "home_team", "home_pts"]].rename(
    columns={"home_team": "team", "home_pts": "pts"}
)
_away_apps = games[["game_id", "season_name", "date", "away_team", "away_pts"]].rename(
    columns={"away_team": "team", "away_pts": "pts"}
)
_apps = (
    pd.concat([_home_apps, _away_apps], ignore_index=True)
    .sort_values(["team", "season_name", "date"])
    .reset_index(drop=True)
)
_apps["season_pts"] = _apps.groupby(["team", "season_name"])["pts"].cumsum()
_apps["game_number"] = _apps.groupby(["team", "season_name"]).cumcount() + 1
_apps["ppg"] = (_apps["season_pts"] / _apps["game_number"]).round(2)
_apps["_date_dt"] = pd.to_datetime(_apps["date"])
_apps["days_rest"] = (
    _apps.groupby(["team", "season_name"])["_date_dt"]
    .diff()
    .dt.days
    .astype("Int64")
)
_apps = _apps.drop(columns=["_date_dt"])

_per_team_cols = ["game_id", "team", "season_pts", "game_number", "days_rest", "ppg"]
games = games.merge(
    _apps[_per_team_cols].rename(columns={
        "team": "home_team",
        "season_pts": "home_season_pts",
        "game_number": "home_game_number",
        "days_rest": "home_days_rest",
        "ppg": "home_ppg",
    }),
    on=["game_id", "home_team"],
    how="left",
)
games = games.merge(
    _apps[_per_team_cols].rename(columns={
        "team": "away_team",
        "season_pts": "away_season_pts",
        "game_number": "away_game_number",
        "days_rest": "away_days_rest",
        "ppg": "away_ppg",
    }),
    on=["game_id", "away_team"],
    how="left",
)

games = games.drop(
    columns=["extra_time", "penalties", "home_penalties", "away_penalties"]
).drop_duplicates().reset_index(drop=True)

In [13]:
# Away team ZIPs are derived from their home appearances in this dataset.
# Teams that appear only as away teams in the filtered date range will have
# no home_zip and will receive null for away_travel_distance.
home_field = (
    games[["home_team", "stadium", "city", "state", "zip"]]
    .drop_duplicates()
    .rename(columns={"home_team": "team"})
    .sort_values(by="team")
    .reset_index(drop=True)
)
zip_lookup = home_field[["team", "zip"]].drop_duplicates(subset="team", keep="first")

games = (
    games.merge(
        zip_lookup,
        left_on="away_team",
        right_on="team",
        how="left",
        suffixes=("", "_lookup"),
    )
    .drop(columns=["team"])
    .rename(columns={"zip_lookup": "away_zip", "zip": "home_zip"})
)

# Precompute zip→coordinates for all unique zips to avoid redundant geocode lookups.
# Non-numeric postal codes (e.g. Canadian teams) are skipped gracefully.
nomi = pgeocode.Nominatim("us")
_zip_coords: dict[str, tuple[float, float] | None] = {}
for _z in pd.concat([games["home_zip"], games["away_zip"]]).dropna().unique():
    try:
        _z_str = str(int(float(_z))).zfill(5)
    except (ValueError, OverflowError):
        continue
    _loc = nomi.query_postal_code(_z_str)
    _zip_coords[_z_str] = (
        (_loc.latitude, _loc.longitude) if not pd.isna(_loc.latitude) else None
    )


def zip_distance_miles(
    zip1: str | float | None, zip2: str | float | None
) -> float | None:
    """Return geodesic distance in miles between two US ZIP codes."""
    if pd.isna(zip1) or pd.isna(zip2):
        return None
    try:
        z1 = str(int(float(zip1))).zfill(5)
        z2 = str(int(float(zip2))).zfill(5)
    except (ValueError, OverflowError):
        return None
    c1 = _zip_coords.get(z1)
    c2 = _zip_coords.get(z2)
    if c1 is None or c2 is None:
        return None
    return geodesic(c1, c2).miles


# Compute distance once per unique (home_zip, away_zip) pair, then merge back.
_pairs = games[["home_zip", "away_zip"]].drop_duplicates()
_pairs["away_travel_distance"] = [
    zip_distance_miles(h, a) for h, a in zip(_pairs["home_zip"], _pairs["away_zip"])
]
games = games.merge(_pairs, on=["home_zip", "away_zip"], how="left")
games["away_travel_distance"] = games["away_travel_distance"].round(0).astype("Int64")

In [14]:
# US state abbreviation → IANA timezone. Enables correct daily weather boundaries
# per venue rather than forcing all venues to America/New_York.
_STATE_TIMEZONE: Final[dict[str, str]] = {
    # Eastern
    "CT": "America/New_York", "DC": "America/New_York", "DE": "America/New_York",
    "FL": "America/New_York", "GA": "America/New_York",
    "IN": "America/Indiana/Indianapolis", "KY": "America/New_York",
    "MA": "America/New_York", "MD": "America/New_York", "ME": "America/New_York",
    "MI": "America/Detroit",  "NH": "America/New_York", "NJ": "America/New_York",
    "NY": "America/New_York", "NC": "America/New_York", "OH": "America/New_York",
    "PA": "America/New_York", "RI": "America/New_York", "SC": "America/New_York",
    "TN": "America/Chicago",  "VT": "America/New_York", "VA": "America/New_York",
    "WV": "America/New_York",
    # Central
    "AL": "America/Chicago", "AR": "America/Chicago", "IA": "America/Chicago",
    "IL": "America/Chicago", "KS": "America/Chicago", "LA": "America/Chicago",
    "MN": "America/Chicago", "MO": "America/Chicago", "MS": "America/Chicago",
    "ND": "America/Chicago", "NE": "America/Chicago", "OK": "America/Chicago",
    "SD": "America/Chicago", "TX": "America/Chicago", "WI": "America/Chicago",
    # Mountain
    "AZ": "America/Phoenix",      "CO": "America/Denver", "ID": "America/Boise",
    "MT": "America/Denver",       "NM": "America/Denver", "UT": "America/Denver",
    "WY": "America/Denver",
    # Pacific
    "CA": "America/Los_Angeles", "NV": "America/Los_Angeles",
    "OR": "America/Los_Angeles", "WA": "America/Los_Angeles",
    # Other US territories (future expansion)
    "AK": "America/Anchorage", "HI": "Pacific/Honolulu",
}

# Map each home ZIP to its venue timezone for accurate daily weather boundaries.
_zip_timezone: dict[str, str] = {}
for _z_val, _state_val in (
    games[["home_zip", "state"]]
    .dropna(subset=["home_zip"])
    .drop_duplicates(subset="home_zip")
    .itertuples(index=False)
):
    try:
        _z_key = str(int(float(_z_val))).zfill(5)
    except (ValueError, OverflowError):
        continue
    _zip_timezone[_z_key] = _STATE_TIMEZONE.get(str(_state_val), "America/New_York")


def fetch_venue_weather(
    lat: float,
    lon: float,
    start_date: str,
    end_date: str,
    timezone: str = "America/New_York",
    *,
    session: requests.Session | None = None,
) -> pd.DataFrame:
    """Fetch daily weather from Open-Meteo archive for a venue's date range.

    Returns temperature (°F), precipitation (inches), and snowfall (inches)
    for each date in the range. Future dates are excluded before the call;
    the caller handles games with no weather data via the left merge.
    """
    resp = (session or requests).get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum",
            "temperature_unit": "fahrenheit",
            "precipitation_unit": "inch",
            "timezone": timezone,
        },
        timeout=15,
    )
    resp.raise_for_status()
    daily = resp.json()["daily"]
    return pd.DataFrame({
        "date": pd.to_datetime(daily["time"]).date,
        "temp_max_f": daily["temperature_2m_max"],
        "temp_min_f": daily["temperature_2m_min"],
        "precip_in": daily["precipitation_sum"],
        "snowfall_in": daily["snowfall_sum"],
    })


today = date.today()
_session = requests.Session()
_weather_frames: list[pd.DataFrame] = []

for _home_zip, _group in games.groupby("home_zip"):
    if pd.isna(_home_zip):
        continue
    try:
        _zip_str = str(int(float(_home_zip))).zfill(5)
    except (ValueError, OverflowError):
        continue
    _coords = _zip_coords.get(_zip_str)
    if _coords is None:
        continue
    _tz = _zip_timezone.get(_zip_str, "America/New_York")
    _past_dates = sorted(d for d in _group["date"].unique() if d <= today)
    if not _past_dates:
        continue
    try:
        _weather = fetch_venue_weather(
            _coords[0], _coords[1],
            str(_past_dates[0]), str(_past_dates[-1]),
            _tz,
            session=_session,
        )
        _weather["home_zip"] = _home_zip
        _weather_frames.append(_weather)
    except requests.RequestException:
        pass

if _weather_frames:
    weather = pd.concat(_weather_frames, ignore_index=True)
    games = games.merge(weather, on=["date", "home_zip"], how="left")

In [15]:
_col_order = [
    # Identity and scheduling
    "game_id", "season_name", "matchday", "date", "day_of_week", "time_est", "date_time_est",
    # Teams and personnel
    "matchup", "home_team", "away_team", "home_manager", "away_manager", "referee",
    # Venue
    "stadium", "city", "state", "home_zip",
    # Score and result
    "home_score", "away_score", "goal_diff", "result", "home_pts", "away_pts",
    "home_season_pts", "away_season_pts", "home_ppg", "away_ppg",
    # xG and xPoints
    "home_team_xgoals", "away_team_xgoals", "xg_diff", "xresult",
    "home_xpoints", "away_xpoints",
    # Match details
    "expanded_minutes", "attendance",
    # Team context
    "home_game_number", "away_game_number", "home_days_rest", "away_days_rest",
    # Travel
    "away_zip", "away_travel_distance",
    # Weather (conditionally present)
    "temp_max_f", "temp_min_f", "precip_in", "snowfall_in",
    # Metadata
    "status", "knockout_game", "last_updated_utc",
]
games = games[[c for c in _col_order if c in games.columns]]
# games.info()
display(games.head())

,game_id,season_name,matchday,date,day_of_week,time_est,date_time_est,matchup,home_team,away_team,home_manager,away_manager,referee,stadium,city,state,home_zip,home_score,away_score,goal_diff,result,home_pts,away_pts,home_season_pts,away_season_pts,home_ppg,away_ppg,home_team_xgoals,away_team_xgoals,xg_diff,xresult,home_xpoints,away_xpoints,expanded_minutes,attendance,home_game_number,away_game_number,home_days_rest,away_days_rest,away_zip,away_travel_distance,temp_max_f,temp_min_f,precip_in,snowfall_in,status,knockout_game,last_updated_utc
0,aDQ0Wby3qE,2026,6,2026-04-08,Wednesday,22:00:00,2026-04-08 22:00:00,OCvSA,OC,SA,Danny Stone,Carlos Llamosa,Natalie Simon,Championship Soccer Stadium,Irvine,California,92618,2,0,2,OC,3,0,11,11,1.83,1.83,2.44,0.94,1.50,OC,2.36,0.48,98,0,6,6,4,4,78233,1173,NaN,NaN,NaN,NaN,FullTime,False,2026-04-09 04:56:37 UTC
1,9YqdejgEQv,2026,5,2026-04-04,Saturday,22:00:00,2026-04-04 22:00:00,MBvSA,MB,SA,Jordan Stewart,Carlos Llamosa,Edson Carvajal,Cardinale Stadium,Seaside,California,93955,0,0,0,DRAW,1,1,2,11,0.4,2.2,0.74,1.04,-0.30,SA,1.05,1.60,96,3430,5,5,7,6,78233,1439,NaN,NaN,NaN,NaN,FullTime,False,2026-04-08 20:27:12 UTC
2,9vQ2GbymMK,2026,5,2026-04-04,Saturday,22:00:00,2026-04-04 22:00:00,SACvPHX,SAC,PHX,Neill Collins,Pa-Modou Kah,Muhammad Hassan,Papa Murphy’s Park,Sacramento,California,95815,2,0,2,SAC,3,0,8,3,1.6,0.6,1.21,1.55,-0.34,PHX,1.08,1.65,101,9308,5,5,7,7,85034,635,NaN,NaN,NaN,NaN,FullTime,False,2026-04-05 05:15:33 UTC
3,KXMeXZGPQ6,2026,5,2026-04-04,Saturday,22:00:00,2026-04-04 22:00:00,OCvNM,OC,NM,Danny Stone,Dennis Sanchez,Thomas Snyder,Championship Soccer Stadium,Irvine,California,92618,0,1,-1,NM,0,3,8,6,1.6,1.5,1.41,0.93,0.48,OC,1.65,1.02,102,3726,5,4,7,7,87106,640,NaN,NaN,NaN,NaN,FullTime,False,2026-04-05 05:35:43 UTC
4,eV5D8p7n5K,2026,5,2026-04-04,Saturday,21:00:00,2026-04-04 21:00:00,ELPvLV,ELP,LV,Junior Gonzalez,Devin Rensing,Ben Meyer,Southwest University Park,El Paso,Texas,79901,3,2,1,ELP,3,0,10,4,2.5,0.8,1.59,1.11,0.48,ELP,1.77,0.96,97,5432,4,5,7,7,89101,582,NaN,NaN,NaN,NaN,FullTime,False,2026-04-05 04:44:42 UTC


In [16]:
games.to_parquet("data/games.parquet", index=False)